In [ ]:
# Install
!pip install streamlit pyngrok pandas plotly --quiet

import os
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = "/content/drive/MyDrive/stock project"

app_code = '''
import streamlit as st
import pandas as pd
import os
import plotly.graph_objects as go

BASE_PATH = "/content/drive/MyDrive/stock project"

st.set_page_config(page_title="Stock Dashboard", layout="centered")

# 🎨 Styling
st.markdown("""
<style>
body {
    background-color: #0b0f19;
}
.big-title {
    font-size:32px !important;
    font-weight: bold;
    color: #00ffcc;
    text-align: center;
}
.card {
    padding: 15px;
    border-radius: 12px;
    margin-bottom: 12px;
    color: white;
    font-weight: 500;
}
.green { background: #0f9d58; }
.red { background: #d32f2f; }
.blue { background: #1e3c72; }
.dark { background: #232526; }
</style>
""", unsafe_allow_html=True)

st.markdown('<p class="big-title">📈 Stock Market Dashboard</p>', unsafe_allow_html=True)

# 📊 Stock selection
stocks = [d for d in os.listdir(BASE_PATH) if os.path.isdir(os.path.join(BASE_PATH, d))]
stock = st.selectbox("Select Stock", stocks)

if stock:
    st.subheader(f"📊 Analysis for {stock.upper()}")

    folder = os.path.join(BASE_PATH, stock)

    file = None
    for f in os.listdir(folder):
        if f.endswith(".csv"):
            file = os.path.join(folder, f)

    if file:
        df = pd.read_csv(file)

        # 🔍 Detect close column
        close_col = None
        for col in df.columns:
            if "close" in col.lower():
                close_col = col

        if close_col:
            # Clean data
            df[close_col] = df[close_col].astype(str).str.replace(',', '')
            df[close_col] = pd.to_numeric(df[close_col], errors='coerce')
            df = df.dropna()

            latest = df[close_col].iloc[-1]
            first = df[close_col].iloc[0]

            # 📈 Market Trend
            if latest > first:
                st.markdown("<div class='card green'>📈 Market Trend: <b>Increasing</b></div>", unsafe_allow_html=True)
            else:
                st.markdown("<div class='card red'>📉 Market Trend: <b>Decreasing</b></div>", unsafe_allow_html=True)

            # 📊 Stability Score
            acc = round(df[close_col].pct_change().mean()*100, 2)
            st.markdown(f"<div class='card blue'>📊 Market Stability Score: <b>{acc}</b></div>", unsafe_allow_html=True)

            # 💰 Buy/Sell
            if acc > 0:
                st.markdown("<div class='card green'>💰 BUY Signal</div>", unsafe_allow_html=True)
            else:
                st.markdown("<div class='card red'>⚠ SELL Signal</div>", unsafe_allow_html=True)

            # 💵 Latest Price
            st.markdown(f"<div class='card dark'>💵 Latest Price: <b>{round(latest,2)}</b></div>", unsafe_allow_html=True)

            # 🔮 Prediction
            predicted = latest * (1 + df[close_col].pct_change().mean())
            st.markdown(f"<div class='card blue'>🔮 Next Day Prediction: <b>{round(predicted,2)}</b></div>", unsafe_allow_html=True)

            # 💹 Candlestick Chart
            fig = go.Figure(data=[go.Candlestick(
                x=df.index,
                open=df[close_col],
                high=df[close_col],
                low=df[close_col],
                close=df[close_col]
            )])

            fig.update_layout(
                template="plotly_dark",
                title="📊 Candlestick Chart",
                height=450
            )

            st.plotly_chart(fig)

        else:
            st.error("❌ No Close column found")

    else:
        st.error("❌ CSV not found")
'''

with open("final_app.py", "w") as f:
    f.write(app_code)

from pyngrok import ngrok
ngrok.kill()
!ngrok authtoken 3BZvshkAY8uy6q0wu7i3Wwo6YFf_6bPhVDnDRgfnYcskwLKer

url = ngrok.connect(8501)
print("🔗 OPEN THIS:", url)

!streamlit run final_app.py &

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 27.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 32.8 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml
🔗 OPEN THIS: NgrokTunnel: "https://lochlan-eversible-justifiably.ngrok-free.dev" -> "http://localhost:8501"



  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.91.81.127:8501

